# Evaluation of similarity based methods

## 1. Import Libraries and Configure Paths

In [241]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from typing import Tuple
from metrics import (
    mean_average_precision_score,
    F1_score_from_sim,
    IoU_from_sim,
    F1_score_q
)

%load_ext autoreload
%autoreload 2

sim_train_path  = Path("/media/eceo_scratch_haas001/results/clip4clip_ft_VE/similarity_matrix_train_6.csv")
sim_test_path   = Path("/media/eceo_scratch_haas001/results/clip4clip_ft_VE/similarity_matrix_test_6.csv")
gt_associations_train_path  = Path("../zero_shot/CLIP4Clip/dataset/train/ground_truth.json")
gt_associations_test_path   = Path("../zero_shot/CLIP4Clip/dataset/test/ground_truth.json")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load Similarity Matrices

In [242]:
# Load raw similarity matrices  shape: (n_videos, n_queries), index = video_id
sim_train_df = pd.read_csv(sim_train_path, index_col="video_id").astype(float)
sim_test_df = pd.read_csv(sim_test_path, index_col="video_id").astype(float)

## Load Ground Truth and Per-Query Thresholds

In [243]:
# Ground truth: {query_text: {"videos": [video_id, ...], "category": "RARE"|"COMMON"}}
with open(gt_associations_test_path, "r") as f:
    gt_associations_test = json.load(f)

print(f"GT queries   : {len(gt_associations_test)}")

# Query categories
with open("../benchmark/query_categories.json", "r") as f:
    query_categories = json.load(f)

query2eco_cat = {q: [] for q in gt_associations_test.keys()}
for cat in query_categories["ECOLOGY"]:
    for q in query_categories["ECOLOGY"][cat]:
        if q in query2eco_cat:
            query2eco_cat[q].append(cat)

query2cv_cat = {q: [] for q in gt_associations_test.keys()}
for cat in query_categories["VISION"]:
    for q in query_categories["VISION"][cat]:
        if q in query2cv_cat:
            query2cv_cat[q].append(cat)

print(f"Ecological Category values      : {set(query_categories['ECOLOGY'].keys())}")
print(f"Vision Category values          : {set(query_categories['VISION'].keys())}")

GT queries   : 113
Ecological Category values      : {'CAMERA_REACTION', 'COMMON', 'RARE', 'SOCIAL', 'COURTSHIP'}
Vision Category values          : {'MULTI_INDIV', 'COMPLEX', 'SINGLE_ATTR', 'VIDEO_COMPARISON', 'MULTI_ATTR'}


In [244]:
# mean Average Precision over queries on the test similarity scores. 
mAP, APs = mean_average_precision_score(gt_associations_test, sim_test_df)
print(f"mAP (test): {mAP:.3f}")

mAP (test): 0.210


## Find Optimal Thresholds on Train Split

In [245]:
with open(gt_associations_train_path, "r") as f:
    gt_associations_train = json.load(f)
queries = gt_associations_train.keys()

In [246]:
thrs = np.arange(sim_train_df.values.min(), sim_train_df.values.max(), step=0.2)
best_t = dict.fromkeys(gt_associations_train.keys())
all_train_videos = list([f.stem for f in Path("/media/EVO870/datasets/prompting-mammalps-v2/annotations/train").rglob("*.json")])

for q in queries:
    best_f1 = 0
    if q not in sim_train_df.columns:
        print(f"Query {q} has not been processed")
        continue
    gt_videos_q = list(gt_associations_train[q]["videos"])
    for t in thrs:
        ass_videos_q = list(sim_train_df.index[sim_train_df[q] > t])
        if len(gt_videos_q) == 0:  # Empty query: measure the opposite
            gt_eff = all_train_videos
            pred_eff = list(set(all_train_videos) - set(ass_videos_q))
        else:
            gt_eff = gt_videos_q
            pred_eff = ass_videos_q
        f1 = F1_score_q(gt_eff, pred_eff)
        if not np.isnan(f1) and f1 > best_f1:
            best_f1 = f1
            best_t[q] = t

    if best_t[q] is None:
        print(f"No best threshold found for {q}, using min similarity score as threshold")
        best_t[q] = thrs[0]

In [247]:
mF1_train, F1_scores_train = F1_score_from_sim(gt_queries_videos=gt_associations_train, sim_matrix_df=sim_train_df, best_thresholds=best_t, all_videos=all_train_videos)

In [248]:
print(f"F1-score (macro-avg. on train set): {mF1_train:.3f}")

F1-score (macro-avg. on train set): 0.660


## Evaluate on Test Split (Thresholds Calibrated on Train)

The per-query thresholds found above are now applied to the **held-out test split**.
This is the only number that should appear in the paper.

In [249]:
# Apply calibrated thresholds to the normalized TEST similarity matrices
all_test_videos = list([f.stem for f in Path("/media/EVO870/datasets/prompting-mammalps-v2/annotations/test").rglob("*.json")])

mF1_test, F1_scores_test = F1_score_from_sim(
    gt_queries_videos=gt_associations_test,
    sim_matrix_df=sim_test_df,
    best_thresholds=best_t,
    all_videos=all_test_videos
)

print("Test performance (thresholds from train):")
print(f"  F1={mF1_test:.3f}")

Test performance (thresholds from train):
  F1=0.262


In [250]:
results_df = pd.concat([
    pd.DataFrame.from_dict(F1_scores_test, orient="index", columns=["F1-score"]),
    ], axis=1)

eco_cat_df = pd.DataFrame([query2eco_cat], index=["eco_cat"]).T
cv_cat_df = pd.DataFrame([query2cv_cat], index=["cv_cat"]).T
results_cat_df = results_df.merge(eco_cat_df, left_index=True, right_index=True).merge(cv_cat_df, left_index=True, right_index=True)

In [251]:
results_cat_df[["F1-score", "eco_cat"]].explode("eco_cat").groupby("eco_cat").mean()

,F1-score
eco_cat,
CAMERA_REACTION,0.183328
COMMON,0.697270
COURTSHIP,0.224634
RARE,0.287007
SOCIAL,0.298896


In [252]:
results_cat_df[["F1-score", "cv_cat"]].explode("cv_cat").groupby("cv_cat").mean()

,F1-score
cv_cat,
COMPLEX,0.253923
MULTI_ATTR,0.253752
MULTI_INDIV,0.196079
SINGLE_ATTR,0.285114
